# Clase 6 — LOB Data Science Pipeline

**Objetivo:** Construir el pipeline completo para modelar variables del libro de órdenes.

**Pregunta central:** Sabes leer el libro. Sabes enviar órdenes. ¿Puedes predecir lo que va a pasar después?

---

En L5 colocaste un limit bid y observaste si se ejecutaba. Hoy vamos a construir el pipeline que convierte esa intuición en un modelo: datos → features → target → split → baseline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

plt.style.use('dark_background')
CYAN   = '#22d3ee'
GREEN  = '#4ade80'
RED    = '#f87171'
MUTED  = '#a1a1aa'

# Cargamos los mismos datos de L4
df = pd.read_csv('../04-market-microstructure-btc/data/btc_lob_snapshots.csv')
print(f'Snapshots cargados: {len(df)} filas, {len(df.columns)} columnas')
df.head(2)

## Paso 1 — Feature Engineering

Un **feature** es una función de los datos crudos que captura una señal útil para el modelo.  
El LOB tiene 40 columnas (10 niveles × bid/ask × price/size). Las convertimos en 8 features interpretables.

In [ ]:
# — Features ya conocidos de L4 —
df['mid']    = (df['bid_price_1'] + df['ask_price_1']) / 2
df['spread'] = df['ask_price_1'] - df['bid_price_1']

bid_vol_5 = sum(df[f'bid_size_{i}'] for i in range(1, 6))
ask_vol_5 = sum(df[f'ask_size_{i}'] for i in range(1, 6))
df['bid_vol_5']  = bid_vol_5
df['ask_vol_5']  = ask_vol_5
df['imbalance']  = df['bid_vol_5'] / (df['bid_vol_5'] + df['ask_vol_5'])

# — Features nuevos —
# Weighted mid: pondera el mid por las sizes del mejor nivel
df['wmid'] = (
    df['bid_price_1'] * df['ask_size_1'] +
    df['ask_price_1'] * df['bid_size_1']
) / (df['bid_size_1'] + df['ask_size_1'])

# Depth ratio: volumen total bid vs ask (10 niveles)
bid_vol_10 = sum(df[f'bid_size_{i}'] for i in range(1, 11))
ask_vol_10 = sum(df[f'ask_size_{i}'] for i in range(1, 11))
df['depth_ratio'] = bid_vol_10 / ask_vol_10

# Spread relativo: spread en % del mid
df['spread_pct'] = df['spread'] / df['mid']

FEATURES = ['mid', 'spread', 'imbalance', 'wmid', 'depth_ratio', 'spread_pct']
print('Features calculados:')
df[FEATURES].describe().round(4)

In [ ]:
# Auditoría visual — siempre mira los datos antes de modelar
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
fig.suptitle('Series temporales de features', color='white', fontsize=13)

colors = [CYAN, GREEN, RED, '#f59e0b', '#a78bfa', MUTED]
for ax, feat, col in zip(axes.flat, FEATURES, colors):
    ax.plot(df[feat].values, color=col, linewidth=0.8)
    ax.set_title(feat, color='white', fontsize=10)
    ax.set_xlabel('snapshot', color=MUTED, fontsize=8)
    ax.tick_params(colors=MUTED)
    for spine in ax.spines.values():
        spine.set_edgecolor('#27272a')
    ax.set_facecolor('#18181b')

fig.patch.set_facecolor('#09090b')
plt.tight_layout()
plt.show()

print('\nObservación: mid y wmid son casi idénticos — wmid captura una micro-corrección.')
print('El imbalance y depth_ratio fluctúan alrededor de 0.5 — mercado razonablemente equilibrado.')

## Paso 2 — Definir el target

El **target** es lo que queremos predecir. La elección del target es una decisión de diseño, no un dato.  

Opciones posibles:
- Precio exacto en t+1 → muy difícil, ruido alto
- Cambio de precio en t+1 → regresión, sensible a outliers  
- **Dirección en t+1** → clasificación binaria, más estable ← usamos esto

Empezamos con la versión más sencilla que aún tiene sentido económico.

In [ ]:
# mid del siguiente snapshot
df['mid_next']   = df['mid'].shift(-1)
df['mid_change'] = df['mid_next'] - df['mid']

# Target binario: 1 = precio sube, 0 = precio baja o igual
df['direction'] = (df['mid_change'] > 0).astype(int)

# Eliminamos la última fila (target = NaN)
df_clean = df.dropna(subset=['mid_next']).reset_index(drop=True)

up   = int((df_clean['direction'] == 1).sum())
down = int((df_clean['direction'] == 0).sum())
print(f'Filas limpias: {len(df_clean)}')
print(f'Subidas (1):  {up}  ({up/len(df_clean)*100:.1f}%)')
print(f'Bajadas (0):  {down}  ({down/len(df_clean)*100:.1f}%)')

# Distribución del cambio de precio
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(df_clean['mid_change'], bins=40, color=CYAN, alpha=0.7, edgecolor='none')
ax.axvline(0, color=RED, linewidth=1.5, linestyle='--', label='0 (sin cambio)')
ax.set_title('Distribución del cambio de mid entre snapshots', color='white')
ax.set_xlabel('Δ mid (USD)', color=MUTED)
ax.tick_params(colors=MUTED)
ax.set_facecolor('#18181b')
fig.patch.set_facecolor('#09090b')
ax.legend(labelcolor='white')
plt.tight_layout()
plt.show()

## Paso 3 — Train/Test split temporal

**Regla de oro:** con series temporales, el test set siempre está en el futuro del training set.

Si mezclas el tiempo (split aleatorio), el modelo puede ver datos futuros durante el entrenamiento.  
Evalúas en evaluación y parece funcionar. En producción, falla.

In [ ]:
# ✅ Split temporal CORRECTO
split_idx = int(len(df_clean) * 0.7)
train_df = df_clean.iloc[:split_idx].copy()
test_df  = df_clean.iloc[split_idx:].copy()

print(f'Train: {len(train_df)} snapshots  (índices 0–{split_idx-1})')
print(f'Test:  {len(test_df)} snapshots  (índices {split_idx}–{len(df_clean)-1})')
print(f'\nÚltimo timestamp train: {train_df["timestamp"].iloc[-1]}')
print(f'Primer timestamp test:  {test_df["timestamp"].iloc[0]}')

# Visualización del split
fig, ax = plt.subplots(figsize=(12, 2.5))
ax.scatter(train_df.index, train_df['mid'], s=4, color=GREEN, alpha=0.7, label=f'Train ({len(train_df)})')
ax.scatter(test_df.index,  test_df['mid'],  s=4, color=RED,   alpha=0.7, label=f'Test  ({len(test_df)})')
ax.axvline(split_idx, color='white', linewidth=1, linestyle='--')
ax.text(split_idx+2, ax.get_ylim()[0]*1.0001, 'frontera temporal', color='white', fontsize=8)
ax.set_title('Split temporal del mid price', color='white')
ax.set_xlabel('índice snapshot', color=MUTED)
ax.tick_params(colors=MUTED)
ax.set_facecolor('#18181b')
fig.patch.set_facecolor('#09090b')
ax.legend(labelcolor='white')
plt.tight_layout()
plt.show()

In [ ]:
# ❌ Split ALEATORIO — qué pasa si usas sklearn por defecto
np.random.seed(42)
idx_shuffled = np.random.permutation(len(df_clean))
train_idx_bad = np.sort(idx_shuffled[:split_idx])   # ← índices en orden para visualizar
test_idx_bad  = np.sort(idx_shuffled[split_idx:])

# ¿Cuántos test points caen ANTES del snapshot 349 (zona de entrenamiento)?
test_in_train_zone = (test_idx_bad < split_idx).sum()
print(f'Con split aleatorio: {test_in_train_zone} de {len(test_idx_bad)} test points'
      f' caen en la zona temporal de entrenamiento.')
print('El modelo entrena con datos del futuro y aprende patrones que no existen en producción.')

fig, ax = plt.subplots(figsize=(12, 2.5))
ax.scatter(train_idx_bad, df_clean.loc[train_idx_bad, 'mid'], s=4, color=GREEN, alpha=0.5, label='Train (aleatorio)')
ax.scatter(test_idx_bad,  df_clean.loc[test_idx_bad,  'mid'], s=4, color=RED,   alpha=0.7, label='Test  (aleatorio)')
ax.axvline(split_idx, color='white', linewidth=1, linestyle='--')
ax.set_title('Split ALEATORIO — test points dispersos por toda la historia', color=RED)
ax.set_xlabel('índice snapshot', color=MUTED)
ax.tick_params(colors=MUTED)
ax.set_facecolor('#18181b')
fig.patch.set_facecolor('#09090b')
ax.legend(labelcolor='white')
plt.tight_layout()
plt.show()

## Paso 4 — Data Leakage

**Leakage:** un feature que contiene información del futuro contamina la evaluación.  
El modelo aprende a predecir el pasado. La accuracy sube artificialmente. En producción, colapsa.

**Test del leakage:** ¿este feature existía en el momento t de la predicción?

In [ ]:
# Feature leaky: usa ask_price del snapshot SIGUIENTE (t+1)
# En el momento t, ese precio todavía no existe.
df_clean = df_clean.copy()
df_clean['realized_spread'] = df_clean['ask_price_1'].shift(-1) - df_clean['bid_price_1']
df_clean = df_clean.dropna(subset=['realized_spread']).reset_index(drop=True)

# Recompute split con datos actualizados (1 fila menos)
split_idx2 = int(len(df_clean) * 0.7)
train2 = df_clean.iloc[:split_idx2]
test2  = df_clean.iloc[split_idx2:]

feats_clean = ['imbalance', 'spread_pct']
feats_leaky = ['imbalance', 'spread_pct', 'realized_spread']

def eval_lr(train, test, feats):
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(train[feats], train['direction'])
    preds = lr.predict(test[feats])
    return (preds == test['direction']).mean()

acc_clean = eval_lr(train2, test2, feats_clean)
acc_leaky = eval_lr(train2, test2, feats_leaky)

print(f'Accuracy SIN feature leaky:  {acc_clean:.1%}')
print(f'Accuracy CON feature leaky:  {acc_leaky:.1%}  ← sube 35 pp de golpe')
print()
print('El salto de accuracy es la señal de alerta.')
print(f'  realized_spread = ask_price_1[t+1] - bid_price_1[t]')
print(f'  ask_price_1[t+1] no existe en el momento de la predicción.')

In [ ]:
# Visualización del leakage
fig, ax = plt.subplots(figsize=(10, 3))

t = 50  # snapshot de ejemplo
# Dibuja timeline
ax.axhline(0.5, color=MUTED, linewidth=1, xmin=0.05, xmax=0.95)
for x, label, col in [(0.15,'t−1',MUTED), (0.4,'t (predicción)',CYAN), (0.65,'t+1 (futuro)',RED), (0.85,'t+2',MUTED)]:
    ax.scatter([x], [0.5], s=120, color=col, zorder=5)
    ax.text(x, 0.58, label, ha='center', color=col, fontsize=9)

# Flecha del leak: t+1 → t
ax.annotate('', xy=(0.4, 0.35), xytext=(0.65, 0.35),
            arrowprops=dict(arrowstyle='->', color=RED, lw=2))
ax.text(0.52, 0.28, 'ask_price_1[t+1]\ncontamina el feature en t', 
        ha='center', color=RED, fontsize=8.5)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('Data Leakage: información del futuro en el feature', color='white')
ax.axis('off')
ax.set_facecolor('#18181b')
fig.patch.set_facecolor('#09090b')
plt.tight_layout()
plt.show()

print('Checklist anti-leakage:')
print('  [ ] ¿El feature usa solo datos disponibles en el momento t?')
print('  [ ] ¿Hay algún .shift(-N) sin justificación explícita?')
print('  [ ] ¿La accuracy sube drásticamente al añadir este feature?')

## Paso 5 — Baselines

Un **baseline** es el suelo mínimo de performance. Si tu modelo no lo supera, no sirve.

Evaluamos tres baselines en el test set (sin leakage):

In [ ]:
# Trabajamos con el split temporal limpio
split_idx = int(len(df_clean) * 0.7)
train_df = df_clean.iloc[:split_idx].copy()
test_df  = df_clean.iloc[split_idx:].copy()
y_test   = test_df['direction'].values

# Baseline 0 — Siempre predecir UP
pred_always_up = np.ones(len(y_test), dtype=int)
acc_always_up  = (pred_always_up == y_test).mean()

# Baseline 1 — Threshold en imbalance
THRESHOLD = 0.6
pred_threshold = (test_df['imbalance'] > THRESHOLD).astype(int).values
acc_threshold  = (pred_threshold == y_test).mean()

# Baseline 2 — Logistic Regression
feats = ['imbalance', 'spread_pct']
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(train_df[feats], train_df['direction'])
pred_lr = lr.predict(test_df[feats])
acc_lr  = (pred_lr == y_test).mean()

results = {
    'Siempre UP':          acc_always_up,
    f'Threshold ({THRESHOLD})': acc_threshold,
    'Logistic Regression': acc_lr,
}

print(f'{'Baseline':<25} {'Accuracy':>10}')
print('-' * 36)
for name, acc in results.items():
    marker = ' ← mejor' if acc == max(results.values()) else ''
    print(f'{name:<25} {acc:>10.1%}{marker}')

In [ ]:
# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
ax1 = axes[0]
names = list(results.keys())
accs  = [v * 100 for v in results.values()]
colors_bar = [MUTED, GREEN, CYAN]
bars = ax1.bar(names, accs, color=colors_bar, alpha=0.85, width=0.5)
ax1.axhline(50, color=RED, linewidth=1, linestyle='--', label='Azar (50%)')
ax1.set_ylim(45, 60)
ax1.set_ylabel('Accuracy (%)', color='white')
ax1.set_title('Comparación de baselines', color='white')
ax1.tick_params(colors=MUTED, axis='y')
ax1.tick_params(colors='white', axis='x', labelsize=8)
for bar, acc in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{acc:.1f}%', ha='center', color='white', fontsize=9, fontweight='bold')
ax1.set_facecolor('#18181b')
ax1.legend(labelcolor='white')
for spine in ax1.spines.values(): spine.set_edgecolor('#27272a')

# Confusion matrix del LR
ax2 = axes[1]
cm = confusion_matrix(y_test, pred_lr)
im = ax2.imshow(cm, cmap='Blues', vmin=0)
ax2.set_xticks([0,1]); ax2.set_yticks([0,1])
ax2.set_xticklabels(['Pred 0\n(DOWN)', 'Pred 1\n(UP)'], color='white')
ax2.set_yticklabels(['Real 0\n(DOWN)', 'Real 1\n(UP)'], color='white')
ax2.set_title('Confusion matrix — Logistic Regression', color='white')
for i in range(2):
    for j in range(2):
        ax2.text(j, i, str(cm[i,j]), ha='center', va='center', color='white', fontsize=14, fontweight='bold')
ax2.set_facecolor('#18181b')

fig.patch.set_facecolor('#09090b')
plt.tight_layout()
plt.show()

print('\nLectura de la confusion matrix en términos de trading:')
print(f'  FP (falso positivo): predijiste UP, bajó → compraste y perdiste')
print(f'  FN (falso negativo): predijiste DOWN, subió → no compraste y te perdiste la subida')

## Ejemplo end-to-end — Pipeline completo en 15 líneas

In [ ]:
# Pipeline completo: carga → features → target → split → modelo → evaluación

raw     = pd.read_csv('../04-market-microstructure-btc/data/btc_lob_snapshots.csv')
raw['mid']        = (raw['bid_price_1'] + raw['ask_price_1']) / 2
raw['spread_pct'] = (raw['ask_price_1'] - raw['bid_price_1']) / raw['mid']
raw['imbalance']  = (sum(raw[f'bid_size_{i}'] for i in range(1,6)) /
                    (sum(raw[f'bid_size_{i}'] for i in range(1,6)) + sum(raw[f'ask_size_{i}'] for i in range(1,6))))
raw['direction']  = (raw['mid'].shift(-1) > raw['mid']).astype(int)
raw = raw.dropna().reset_index(drop=True)

n = len(raw)
split = int(n * 0.7)
X_train, y_train = raw.iloc[:split][['imbalance', 'spread_pct']], raw.iloc[:split]['direction']
X_test,  y_test  = raw.iloc[split:][['imbalance', 'spread_pct']], raw.iloc[split:]['direction']

model = LogisticRegression(random_state=42, max_iter=1000).fit(X_train, y_train)
preds = model.predict(X_test)
acc   = (preds == y_test.values).mean()
print(f'Accuracy del pipeline: {acc:.1%}')

# Simulación sobre los últimos 10 snapshots del test set
print(f'\n{"Snapshot":<10} {"Imbalance":<12} {"Spread%":<10} {"Predicción":<12} {"Real":<8} {"✓"}')
print('-' * 60)
for i, (idx, row) in enumerate(X_test.tail(10).iterrows()):
    pred = model.predict([row.values])[0]
    real = y_test.loc[idx]
    ok   = '✓' if pred == real else '✗'
    print(f'{idx:<10} {row["imbalance"]:<12.4f} {row["spread_pct"]:.6f}  {"UP" if pred else "DOWN":<12} {"UP" if real else "DOWN":<8} {ok}')

## Cierre — 3 ideas para llevarse

1. **El pipeline importa más que el modelo.** Un modelo potente con un split o un target mal definido produce resultados inútiles.

2. **El split debe respetar el tiempo.** En series temporales, el test siempre está en el futuro del train. `train_test_split` de sklearn sin `shuffle=False` es un error.

3. **El leakage sube la accuracy artificialmente.** Si añadir un feature mejora el resultado de forma inesperada, pregúntate: ¿este dato existía en el momento de la predicción?

---

**Puente a L7:** En L7 aplicamos exactamente este mismo pipeline a tres escenarios con más features y modelos más complejos. La estructura es la misma. Los resultados cambian.

La pregunta que quedó abierta: la LR apenas supera el azar (~50%). ¿Necesitamos mejores features, más datos, o un modelo distinto? En L7 lo averiguamos.